<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/OpenWeightModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hugging Face Login

In [1]:
!pip install -q huggingface_hub

In [2]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## Load the data

In [3]:
!pip install -q datasets

In [4]:
from datasets import load_dataset

raw_dataset = load_dataset("sgoel9/paul_graham_essays", split="train")

print(raw_dataset[0])

{'id': 0, 'title': 'The Age of the Essay', 'date': 'September 2004', 'text': 'Remember the essays you had to write in high school? Topic sentence, introductory paragraph, supporting paragraphs, conclusion. The conclusion being, say, that Ahab in _Moby Dick_ was a Christ-like figure.\n\nOy. So I\'m going to try to give the other side of the story: what an essay really is, and how you write one. Or at least, how I write one.\n\n**Mods**\n\nThe most obvious difference between real essays and the things one has to write in school is that real essays are not exclusively about English literature. Certainly schools should teach students how to write. But due to a series of historical accidents the teaching of writing has gotten mixed together with the study of literature. And so all over the country students are writing not about how a baseball team with a small budget might compete with the Yankees, or the role of color in fashion, or what constitutes a good dessert, but about symbolism in D

## Structure the data

In [5]:
from datasets import Dataset

# Convert list into a Dataset object.
# It is needed so we can split it into train/test parts later.
dataset = Dataset.from_list([
    {
        "text": f"Title: {row['title']}\n\nEssay:\n{row['text']}"
    }
    for row in raw_dataset
])

print(dataset[1])

{'text': 'Title: A Plan for Spam\n\nEssay:\n_(This article describes the spam-filtering techniques used in the spamproof web-based mail reader we built to exercise [Arc](arc.html). An improved algorithm is described in [Better Bayesian Filtering](better.html).)_\n\nI think it\'s possible to stop spam, and that content-based filters are the way to do it. The Achilles heel of the spammers is their message. They can circumvent any other barrier you set up. They have so far, at least. But they have to deliver their message, whatever it is. If we can write software that recognizes their messages, there is no way they can get around that.\n\n\\_ \\_ \\_\n\nTo the recipient, spam is easily recognizable. If you hired someone to read your mail and discard the spam, they would have little trouble doing it. How much do we have to do, short of AI, to automate this process?\n\nI think we will be able to solve the problem with fairly simple algorithms. In fact, I\'ve found that you can filter presen

## Split the data into test and train sets

In [6]:
# seed fixes the random split so it's reproducible
dataset = dataset.train_test_split(test_size=0.1, seed = 42)

## Load and quantize the base model

In [7]:
!pip install -q transformers

In [8]:
from transformers import AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct" # "meta-llama/Llama-3.2-3B-Instruct"

# load the tokenizer that matches the base model, using the HF token since it's gated
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)

# Llama has no pad token by default, so reuse the end-of-sequence token for padding
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [9]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.3 MB/s eta 0:00:00


In [10]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Sets up 4 bit loading which makes QLoRA fit on a T4.
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# Load the model from pretrained weights in quantized form and load it on a GPU.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    # quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

## Configure LoRA

In [16]:
!pip install -q peft -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.8 MB/s eta 0:00:00


In [17]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


In [18]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.3 MB/s eta 0:00:00


In [19]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./pg-essay-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
    report_to="none",
)

In [20]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/193 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/22 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/22 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,2.740338
20,2.668786
30,2.617441
40,2.580643
50,2.577901
60,2.531418
70,2.527491


TrainOutput(global_step=75, training_loss=2.6054645284016926, metrics={'train_runtime': 2331.4871, 'train_samples_per_second': 0.248, 'train_steps_per_second': 0.032, 'total_flos': 3404302344167424.0, 'train_loss': 2.6054645284016926, 'entropy': 2.5923611136043774, 'num_tokens': 533577.0, 'mean_token_accuracy': 0.4502791569513433, 'epoch': 3.0})

In [ ]:
trainer.save_model("./pg-essay-lora-final")

In [1]:
prompt = "Title: How to Start a Startup\n\nEssay:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
print(tokenizer.decode(output[0], skip_special_tokens=True))

NameError: name 'tokenizer' is not defined